In [ ]:
import os
import math
import pandas as pd
import numpy as np
from pathlib import Path
import xml.dom.minidom 
from datetime import datetime
import xml.etree.ElementTree as Et

def get_xml_root():
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    return os.path.join(BASE_DIR, "..", "OhioT1DM")

def get_XMLs(root):
    p = Path(root)
    lista_XML = []

    for x in p.iterdir():
        if x.is_dir() :
            lista_XML.extend(get_XMLs(x))
        # elif "testing" in x.name: 
        #     continue
        # elif "training" in x.name: 
        #     continue
        elif x.suffix == ".xml": 
            lista_XML.append(x)

    return lista_XML

def get_info(file_XML, dados: dict):
    domtree = xml.dom.minidom.parse(str(file_XML))

    patient = domtree.documentElement
    assert patient is not None

    id_patient = int(patient.getAttribute('id'))

    if id_patient not in dados.keys():
        dados[id_patient] = {}
        

    glucose_level = patient.getElementsByTagName('glucose_level')[0].getElementsByTagName('event')
    finger_stick = patient.getElementsByTagName('finger_stick')[0].getElementsByTagName('event')
    basal = patient.getElementsByTagName('basal')[0].getElementsByTagName('event')
    bolus = patient.getElementsByTagName('bolus')[0].getElementsByTagName('event')
    meal = patient.getElementsByTagName('meal')[0].getElementsByTagName('event')
    sleep = patient.getElementsByTagName('sleep')[0].getElementsByTagName('event')
    exercise = patient.getElementsByTagName('exercise')[0].getElementsByTagName('event')


    for event in finger_stick:
        ts =  binning(event.getAttribute('ts'))
        dados[id_patient][ts] = {"medidor": "paciente", 
                                 "glucose_level": int(event.getAttribute('value')),
                                 "basal": None,
                                 "bolus": None,
                                 "bolus_bwz_carb_input": None,
                                 "meal_type": None, 
                                 "meal_carbs": None,
                                 "sleep": None,
                                 "sleep_quality": None,
                                 "exercise_intensity": None,
                                 "glucose_level": None}

    for event in glucose_level:
        ts =  binning(event.getAttribute('ts'))
        try:
            dados[id_patient][ts]["medidor"] = "CGM"
            dados[id_patient][ts]["glucose_level"] = int(event.getAttribute('value'))           
        except:
            dados[id_patient][ts] = {"medidor": "CGM", 
                                     "glucose_level": int(event.getAttribute('value')),
                                     "basal": None,
                                     "bolus": None,
                                     "bolus_bwz_carb_input": None,
                                     "meal_type": None, 
                                     "meal_carbs": None,
                                     "sleep": None,
                                     "sleep_quality": None,
                                     "exercise_intensity": None,
                                     "glucose_level": None}

    for event in basal:
        ts =  binning(event.getAttribute('ts'))
        try:
            dados[id_patient][ts]["basal"] = float(event.getAttribute('value'))
        except:
            dados[id_patient][ts] = {"medidor": None, 
                                     "glucose_level": None,
                                     "basal": float(event.getAttribute('value')),
                                     "bolus": None,
                                     "bolus_bwz_carb_input": None,
                                     "meal_type": None, 
                                     "meal_carbs": None,
                                     "sleep": None,
                                     "sleep_quality": None,
                                     "exercise_intensity": None,
                                     "glucose_level": None}

    for event in bolus:
        ts =  binning(event.getAttribute('ts_begin'))
        try:
            dados[id_patient][ts]["bolus"] = event.getAttribute('dose')
            dados[id_patient][ts]["bolus_bwz_carb_input"] = event.getAttribute('bwz_carb_input')
        except:
            dados[id_patient][ts] = {"medidor": None, 
                                     "glucose_level": None,
                                     "basal": None,
                                     "bolus": event.getAttribute('dose'),
                                     "bolus_bwz_carb_input": event.getAttribute('bwz_carb_input'),
                                     "meal_type": None, 
                                     "meal_carbs": None,
                                     "sleep": None,
                                     "sleep_quality": None,
                                     "exercise_intensity": None,
                                     "glucose_level": None}

    for event in meal:
        ts =  binning(event.getAttribute('ts'))
        try:
            dados[id_patient][ts]["meal_type"] = event.getAttribute('type')
            dados[id_patient][ts]["meal_carbs"] = event.getAttribute('carbs')
        except:
            dados[id_patient][ts] = {"medidor": None, 
                                     "glucose_level": None,
                                     "basal": None,
                                     "bolus": None,
                                     "bolus_bwz_carb_input": None,
                                     "meal_type": event.getAttribute('type'), 
                                     "meal_carbs": event.getAttribute('carbs'),
                                     "sleep": None,
                                     "sleep_quality": None,
                                     "exercise_intensity": None,
                                     "glucose_level": None}

    for event in sleep:
        ts_begin =  binning(event.getAttribute('ts_begin'))
        ts_end =  binning(event.getAttribute('ts_end'))
        try:
            dados[id_patient][ts_begin]["sleep"] = 'comeso'
            dados[id_patient][ts_begin]["sleep_quality"] = event.getAttribute('quality')
        except:
            dados[id_patient][ts_begin] = {"medidor": None, 
                                             "glucose_level": None,
                                             "basal": None,
                                             "bolus": None,
                                             "bolus_bwz_carb_input": None,
                                             "meal_type": None,
                                             "meal_carbs": None,
                                             "sleep": 'comeso',
                                             "sleep_quality": event.getAttribute('quality'),
                                             "exercise_intensity": None,
                                             "glucose_level": None}
        try:
            dados[id_patient][ts_end]["sleep"] = 'fim'
            dados[id_patient][ts_end]["sleep_quality"] = event.getAttribute('quality')
        except:
            dados[id_patient][ts_end] = {"medidor": None, 
                                             "glucose_level": None,
                                             "basal": None,
                                             "bolus": None,
                                             "bolus_bwz_carb_input": None,
                                             "meal_type": None,
                                             "meal_carbs": None,
                                             "sleep": 'fim',
                                             "sleep_quality": event.getAttribute('quality'),
                                             "exercise_intensity": None,
                                             "glucose_level": None}

    for event in exercise:
        ts =  binning(event.getAttribute('ts'))
        try:
            dados[id_patient][ts]["exercise_intensity"] = event.getAttribute('intensity')
            dados[id_patient][ts]["exercise_duration"] = event.getAttribute('duration')
        except:
            dados[id_patient][ts] = {"medidor": None, 
                                     "glucose_level": None,
                                     "basal": None,
                                     "bolus": None,
                                     "bolus_bwz_carb_input": None,
                                     "meal_type": None,
                                     "meal_carbs": None,
                                     "sleep": None,
                                     "sleep_quality": None,
                                     "exercise_intensity": event.getAttribute('intensity'),
                                     "exercise_duration": event.getAttribute('duration')}

    
    return dados

def binning(ts): 
    data = datetime.strptime(ts, "%d-%m-%Y %H:%M:%S")
    mim = math.floor(data.minute/5)*5
    return data.replace(minute = mim, second = 0)
    
lista_XML = get_XMLs(get_xml_root())

dados = {}
for file in lista_XML:
    dados = get_info(file, dados)

df_paciente = pd.DataFrame(dados)
display(df_paciente)

,559,563,570,575,588,591,540,544,552,567,584,596
2022-01-18 06:25:00,"{'medidor': 'CGM', 'glucose_level': 249, 'basa...",NaN,"{'medidor': 'paciente', 'glucose_level': None,...",NaN,NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,NaN,NaN,NaN,NaN
2022-01-18 16:55:00,"{'medidor': 'CGM', 'glucose_level': 150, 'basa...",NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,NaN,NaN,NaN,NaN
2022-01-18 17:55:00,"{'medidor': 'paciente', 'glucose_level': None,...",NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,NaN,NaN,NaN,NaN
2022-01-19 04:30:00,"{'medidor': 'CGM', 'glucose_level': 96, 'basal...",NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,NaN,NaN,NaN,NaN
2022-01-19 05:30:00,"{'medidor': 'CGM', 'glucose_level': 161, 'basa...",NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,"{'medidor': 'CGM', 'glucose_level': None, 'bas...",NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
2027-04-20 08:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'medidor': None, 'glucose_level': None, 'basa..."
2027-04-25 11:40:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'medidor': None, 'glucose_level': None, 'basa..."
2027-05-02 14:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'medidor': None, 'glucose_level': None, 'basa..."
2027-05-02 18:35:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"{'medidor': None, 'glucose_level': None, 'basa..."
